In [1]:
# =============================================================================
# EXTRACTION AUTOMATISÉE NASA POWER API — DONNÉES CLIMATIQUES JOURNALIÈRES
# Google Colab — copier-coller ce script dans une cellule unique
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# BLOC 1 — Installation & imports
# ─────────────────────────────────────────────────────────────────────────────
!pip install pandas requests numpy --quiet

import pandas as pd
import numpy as np
import io
import os
import requests
import time
import calendar
from datetime import datetime
from google.colab import files

print("✅ Librairies chargées.")


# ─────────────────────────────────────────────────────────────────────────────
# BLOC 2 — Chargement du fichier stations_summary.csv
# Colonnes attendues : nom, lat, lon, depuis
# ─────────────────────────────────────────────────────────────────────────────
print("\n📂 Téléchargez votre fichier stations_summary.csv :")
uploaded = files.upload()

if not uploaded:
    raise FileNotFoundError("❌ Aucun fichier téléchargé. Relancez ce bloc.")

filename = list(uploaded.keys())[0]
df_stations = pd.read_csv(io.StringIO(uploaded[filename].decode('utf-8')))

# ── Validation des colonnes minimales ────────────────────────────────────────
COLS_REQUISES = {'nom', 'lat', 'lon', 'depuis'}
cols_csv = set(df_stations.columns.str.strip().str.lower())
manquantes = COLS_REQUISES - cols_csv
if manquantes:
    raise ValueError(f"❌ Colonnes manquantes dans le CSV : {manquantes}\n"
                     f"   Colonnes trouvées : {list(df_stations.columns)}")

# Normalisation des noms de colonnes (minuscules, sans espaces)
df_stations.columns = df_stations.columns.str.strip().str.lower()

# ── Construction du dictionnaire REGIONS ─────────────────────────────────────
REGIONS = {}
for i, row in df_stations.iterrows():
    rid = f'R{i+1:02d}'
    REGIONS[rid] = {
        'nom'   : str(row['nom']).strip(),
        'lat'   : float(row['lat']),
        'lon'   : float(row['lon']),
        'depuis': int(row['depuis']),
    }

print(f"\n✅ {len(REGIONS)} station(s) chargée(s) :\n")
for rid, info in REGIONS.items():
    print(f"  {rid} | {info['nom']:20s} | lat={info['lat']:.4f}  "
          f"lon={info['lon']:.4f}  depuis={info['depuis']}")


# ─────────────────────────────────────────────────────────────────────────────
# BLOC 3 — Paramètres d'extraction NASA POWER
# ─────────────────────────────────────────────────────────────────────────────

ANNEE_FIN = 2024        # Année de fin (NASA POWER a ~1-2 mois de latence)

# URL de l'API NASA POWER v2 (la v1 est obsolète et renvoie 404)
API_NASA_POWER = "https://power.larc.nasa.gov/api/temporal/daily/point" # Changed to daily endpoint

# Paramètres NASA POWER à extraire
# ─────────────────────────────────────────────────────────────────────────────
# T2M            : Température moyenne à 2m (°C)
# T2M_MAX        : Température maximale à 2m (°C)
# T2M_MIN        : Température minimale à 2m (°C)
# T2MDEW         : Température du point de rosée à 2m (°C)
# RH2M           : Humidité relative à 2m (%)
# PRECTOTCORR    : Précipitations totales corrigées (mm/jour)
# WS2M           : Vitesse du vent à 2m (m/s)
# WS10M          : Vitesse du vent à 10m (m/s)
# ALLSKY_SFC_SW_DWN : Rayonnement solaire descendant (MJ/m²/jour)
# PS             : Pression de surface (kPa)
# ─────────────────────────────────────────────────────────────────────────────

NASA_PARAMS = [
    'T2M',
    'T2M_MAX',
    'T2M_MIN',
    'T2MDEW',
    'RH2M',
    'PRECTOTCORR',
    'WS2M',
    'WS10M',
    'ALLSKY_SFC_SW_DWN',
    'PS',
]

# Délais pour éviter le rate limiting
DELAI_ENTRE_STATIONS = 3   # secondes entre chaque station
MAX_RETRIES = 5             # tentatives max par requête
TIMEOUT = 60                # timeout requête en secondes

print(f"\n⚙️  Extraction NASA POWER API v2 — jusqu'à {ANNEE_FIN}")
print(f"   Paramètres : {NASA_PARAMS}")
print(f"   Endpoint   : {API_NASA_POWER}")


# ─────────────────────────────────────────────────────────────────────────────
# BLOC 4 — Fonction d'extraction journalière NASA POWER pour un point GPS
# ─────────────────────────────────────────────────────────────────────────────

session = requests.Session()

def extraire_journalier_nasa_power(region_id, lat, lon, annee_debut, annee_fin=ANNEE_FIN):
    """
    Extrait les données climatiques journalières de NASA POWER pour un point GPS.
    Retourne un DataFrame avec une ligne par jour.

    Paramètres
    ----------
    region_id   : identifiant de la station (str)
    lat, lon    : coordonnées GPS (float)
    annee_debut : première année à extraire (int)
    annee_fin   : dernière année inclusive (int)

    Retour
    ------
    pd.DataFrame  colonnes : datetime, region_id, np_T2M, np_T2M_MAX, ...
    """

    params = {
        'parameters'  : ','.join(NASA_PARAMS),
        'community'   : 'AG',
        'longitude'   : round(lon, 4),
        'latitude'    : round(lat, 4),
        'start'       : f"{annee_debut}0101", # API expects YYYYMMDD for daily
        'end'         : f"{annee_fin}1231",   # API expects YYYYMMDD for daily
        'format'      : 'JSON',
    }

    # ── Requête avec retry ───────────────────────────────────────────────────
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(
                API_NASA_POWER,
                params=params,
                timeout=TIMEOUT,
            )

            # Rate limiting
            if response.status_code == 429:
                wait = 10 * attempt
                print(f"\n   ⏳ Rate limit — attente {wait}s (tentative {attempt}/{MAX_RETRIES})...",
                      end='', flush=True)
                time.sleep(wait)
                continue

            response.raise_for_status()
            data = response.json()

            # ── Vérification de la structure de la réponse ───────────────────
            if 'properties' not in data:
                print(f"\n   ⚠️  {region_id} : clé 'properties' absente")
                return pd.DataFrame()

            props = data['properties']

            if 'parameter' not in props:
                print(f"\n   ⚠️  {region_id} : clé 'parameter' absente")
                return pd.DataFrame()

            param_data = props['parameter']

            # ── Construction du DataFrame ────────────────────────────────────
            rows = []

            # Récupérer les clés temporelles depuis le premier paramètre dispo
            first_param = list(param_data.keys())[0]
            time_keys = sorted(param_data[first_param].keys())

            for tk in time_keys:
                # Format des clés : "YYYYMMDD"
                if len(tk) != 8:
                    continue

                year_str  = tk[:4]
                month_str = tk[4:6]
                day_str   = tk[6:8]

                try:
                    year  = int(year_str)
                    month = int(month_str)
                    day   = int(day_str)
                except ValueError:
                    continue

                # Ignorer les dates hors plage (API devrait déjà le faire mais sécurité)
                if year < annee_debut or year > annee_fin:
                    continue

                row = {
                    'datetime'  : pd.Timestamp(year, month, day),
                    'region_id' : region_id,
                }

                for p_name in NASA_PARAMS:
                    if p_name in param_data and tk in param_data[p_name]:
                        val = param_data[p_name][tk]
                        # NASA POWER utilise -999.0 comme valeur manquante
                        row[f'np_{p_name}'] = val if val != -999.0 else np.nan
                    else:
                        row[f'np_{p_name}'] = np.nan

                rows.append(row)

            return pd.DataFrame(rows)

        except requests.exceptions.Timeout:
            wait = 5 * attempt
            print(f"\n   ⏱️  Timeout — attente {wait}s (tentative {attempt}/{MAX_RETRIES})...",
                  end='', flush=True)
            time.sleep(wait)

        except requests.exceptions.ConnectionError:
            wait = 10 * attempt
            print(f"\n   🔌 Erreur réseau — attente {wait}s (tentative {attempt}/{MAX_RETRIES})...",
                  end='', flush=True)
            time.sleep(wait)

        except Exception as e:
            print(f"\n   ❌ {region_id} erreur inattendue : {e}")
            return pd.DataFrame()

    print(f"\n   ❌ {region_id} : échec après {MAX_RETRIES} tentatives")
    return pd.DataFrame()


# ─────────────────────────────────────────────────────────────────────────────
# BLOC 5 — Lancement de l'extraction pour toutes les stations
# ─────────────────────────────────────────────────────────────────────────────

dfs = []

for i, (rid, info) in enumerate(REGIONS.items()):
    print(f"\n🛰  [{i+1}/{len(REGIONS)}] Extraction NASA POWER — {rid} : {info['nom']} "
          f"(lat={info['lat']:.4f}, lon={info['lon']:.4f}, depuis={info['depuis']})")

    df_region = extraire_journalier_nasa_power( # Changed function call
        region_id   = rid,
        lat         = info['lat'],
        lon         = info['lon'],
        annee_debut = info['depuis'],
        annee_fin   = ANNEE_FIN,
    )

    if len(df_region) > 0:
        # Ajout des métadonnées de station
        df_region['region_nom'] = info['nom']
        df_region['lat']        = info['lat']
        df_region['lon']        = info['lon']
        dfs.append(df_region)
        print(f"   ✅ {len(df_region)} jours extraits pour {info['nom']}") # Changed 'mois' to 'jours'
    else:
        print(f"   ❌ Aucune donnée pour {info['nom']}")

    # Pause entre stations
    if i < len(REGIONS) - 1:
        print(f"   ⏳ Pause {DELAI_ENTRE_STATIONS}s...", end='', flush=True)
        time.sleep(DELAI_ENTRE_STATIONS)
        print(" OK")

if not dfs:
    raise Exception("❌ Aucune donnée extraite pour aucune station.")

df_final = pd.concat(dfs, ignore_index=True)
df_final = df_final.sort_values(['region_nom', 'datetime']).reset_index(drop=True)
print(f"\n✅ Extraction terminée — {len(df_final)} lignes au total.")


# ─────────────────────────────────────────────────────────────────────────────
# BLOC 6 — Post-traitement : conversions d'unités NASA POWER
# ─────────────────────────────────────────────────────────────────────────────

# Précipitations (déjà en mm/jour) et Rayonnement (déjà en MJ/m²/jour)
# Aucune conversion mensuelle nécessaire pour les données journalières.

# ── Pression : kPa → hPa ────────────────────────────────────────────────────
if 'np_PS' in df_final.columns:
    df_final['pressure_hPa'] = df_final['np_PS'] * 10

# ── Vitesse du vent : déjà en m/s ───────────────────────────────────────────
# np_WS2M  → vent à 2m  (m/s)
# np_WS10M → vent à 10m (m/s)

# ── Colonnes renommées pour lisibilité ───────────────────────────────────────
RENOMMAGE = {
    'np_T2M'               : 'temp_mean_C',
    'np_T2M_MAX'           : 'temp_max_C',
    'np_T2M_MIN'           : 'temp_min_C',
    'np_T2MDEW'            : 'dewpoint_C',
    'np_RH2M'              : 'humidity_pct',
    'np_WS2M'              : 'wind_2m_ms',
    'np_WS10M'             : 'wind_10m_ms',
    'np_PRECTOTCORR'       : 'precip_mm_jour',
    'np_ALLSKY_SFC_SW_DWN' : 'radiation_MJm2_jour',
    'np_PS'                : 'pressure_kPa',
}

df_final = df_final.rename(columns=RENOMMAGE)

print("✅ Conversions d'unités appliquées.")
print("\n📋 Colonnes finales :")
for c in df_final.columns:
    print(f"   • {c}")


# ─────────────────────────────────────────────────────────────────────────────
# BLOC 7 — Aperçu des données
# ─────────────────────────────────────────────────────────────────────────────

print("\n📊 Aperçu des premières lignes :")
print(df_final.head(10).to_string(index=False))

print("\n📊 Statistiques descriptives :")
cols_num = df_final.select_dtypes(include=[np.number]).columns.tolist()
cols_num = [c for c in cols_num if c not in ['lat', 'lon']]
print(df_final[cols_num].describe().round(2).to_string())


# ─────────────────────────────────────────────────────────────────────────────
# BLOC 8 — Sauvegarde et téléchargement
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_CSV = f'extraction_nasa_power_quotidienne_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv' # Updated filename

df_final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f"\n💾 Fichier sauvegardé : {OUTPUT_CSV}")
print(f"   {df_final.shape[0]} lignes × {df_final.shape[1]} colonnes")

# Téléchargement automatique dans le navigateur
files.download(OUTPUT_CSV)
print("⬇️  Téléchargement lancé.")


# ─────────────────────────────────────────────────────────────────────────────
# BLOC 9 — Résumé statistique par station
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*80)
print("📊 RÉSUMÉ PAR STATION (Moyennes Journalières)") # Updated title
print("="*80)

resume = df_final.groupby('region_nom').agg({
    'datetime'           : ['min', 'max', 'count'],
    'temp_mean_C'        : ['mean', 'min', 'max'],
    'precip_mm_jour'     : 'mean', # Changed to mean for daily data
    'humidity_pct'       : 'mean',
    'wind_10m_ms'        : 'mean',
    'radiation_MJm2_jour': 'mean', # Changed to mean for daily data
}).round(2)

resume.columns = [
    'Début', 'Fin', 'Nb_jours', # Changed 'Nb_mois' to 'Nb_jours'
    'T_moy_°C', 'T_min_°C', 'T_max_°C',
    'P_moy_mm/j', # Removed 'P_tot_mm', kept daily mean
    'HR_moy_%', 'Vent_10m_m/s', 'Rad_MJ/m²/j',
]

print(resume.to_string())

print("\n✅ Extraction NASA POWER terminée avec succès !")

✅ Librairies chargées.

📂 Téléchargez votre fichier stations_summary.csv :


Saving stations_summary.csv to stations_summary.csv

✅ 24 station(s) chargée(s) :

  R01 | ksar El Kebir        | lat=35.0035  lon=-5.9155  depuis=2012
  R02 | DAR ELGUEDDARI       | lat=34.4113  lon=-6.1101  depuis=2012
  R03 | SIDI ALLAL TAZI      | lat=34.5220  lon=-6.3428  depuis=2015
  R04 | SUTA - FERME ABT     | lat=32.2123  lon=-6.7878  depuis=2015
  R05 | SUTA-TAZEROUALT      | lat=32.2168  lon=-6.7972  depuis=2016
  R06 | SUTA OULAD AYAD      | lat=32.2123  lon=-6.7878  depuis=2015
  R07 | MECHRAA BELKSIRI     | lat=34.5740  lon=-5.9515  depuis=2018
  R08 | SUTA 507             | lat=32.4356  lon=-6.7510  depuis=2019
  R09 | M-Zaio               | lat=34.9275  lon=-2.7537  depuis=2016
  R10 | Merksen              | lat=32.4357  lon=-6.7510  depuis=2017
  R11 | SUTA_CENTAGRI        | lat=32.3411  lon=-6.9802  depuis=2017
  R12 | SUTA INRA            | lat=32.2604  lon=-6.5250  depuis=2018
  R13 | SUTA 503             | lat=32.5127  lon=-6.5726  depuis=2018
  R14 | SUTA 505    

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Téléchargement lancé.

📊 RÉSUMÉ PAR STATION (Moyennes Journalières)
                      Début        Fin  Nb_jours  T_moy_°C  T_min_°C  T_max_°C  P_moy_mm/j  HR_moy_%  Vent_10m_m/s  Rad_MJ/m²/j
region_nom                                                                                                                     
DAR ELGUEDDARI   2012-01-01 2024-12-31      4749     19.74      6.55     37.02        1.19     69.15          3.78        18.61
DK-FAREGH 310    2019-01-01 2024-12-31      2192     19.31      8.58     34.15        1.12     69.30          3.87        20.26
DK-GHARBIA 343   2019-01-01 2024-12-31      2192     19.32      8.36     34.43        0.86     68.42          4.13        20.26
DK-SEMVAD        2019-01-01 2024-12-31      2192     19.32      8.36     34.43        0.86     68.42          4.13        20.26
DK-USINE SB      2019-01-01 2024-12-31      2192     19.32      8.36     34.43        0.86     68.42          4.13        20.26
DK-USINE ZEMAMRA 2019-01-01 2024